In [1]:
import pandas as pd
# read the sentence data 
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/sentence_data_for_analysis.xlsx", index_col=0)
sentences = df["sentence"].to_list()

# check the df head
df.head()

,message_id,sentence,clean_sentence,sentence_id,translated_sentence
0,1,"Geachte ibd groep, Is mijn uitslag al binnen ...","ibd groep, Is mijn uitslag al binnen van de b...",1,"Dear Ibd group, has my results come back from ..."
1,3,Vorige week is door [ZIEKENHUIS] [LOCATIE] mij...,Vorige week is door mijn ontlasting onderzoc...,2,Last week my stool was examined by [SIGHSHOUSE...
2,3,Graag zou ik de uitkomst hiervan vernemen.,Graag zou ik de uitkomst hiervan vernemen.,3,I would like to hear the outcome of this.
3,4,bloed in de ontlasting wordt steeds meer en st...,bloed in de ontlasting wordt steeds meer en st...,4,blood in the stool is becoming more and more f...
4,4,Ligt dit aan de medicatie?,Ligt dit aan de medicatie?,5,Is this because of the medication?


In [3]:
### Importing the list of Dutch stopwords (note that there are customized dutch words in there)

with open('/workspace/persistent/mijnidbcoachnlp/data/analysis_data/stopwords_extended.txt', 'r') as file:
    lines = [line.strip() for line in file.readlines()]

dutch_stopwords = lines

In [4]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired
import numpy as np
from sentence_transformers import SentenceTransformer

# Shared settings (avoid code duplication)
bertopic_settings = {
    "umap_model": UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42), # fixed random_state for reproducibility
    "hdbscan_model": HDBSCAN(min_cluster_size=25, metric='euclidean', cluster_selection_method='eom', prediction_data=False),
    "vectorizer_model": CountVectorizer(stop_words=dutch_stopwords, min_df=2, ngram_range=(1, 1), token_pattern=r'\b[a-zA-Z]{3,}\b'),
    "calculate_probabilities": False,
    "verbose": False
    #"representation_model": { 
    #    "KeyBERTInspired": KeyBERTInspired()
        #...
    #}
}

embedding_model = SentenceTransformer("NetherlandsForensicInstitute/robbert-2022-dutch-sentence-transformers")
embeddings = np.load("/workspace/persistent/mijnidbcoachnlp/data/embeddings/embeddings_st_robbert2022_sentence_placeholder.npy")

In [6]:
from bertopic import BERTopic

topic_model = BERTopic(**bertopic_settings)
topics, probs = topic_model.fit_transform(sentences, embeddings)

In [7]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,12933,-1_medicatie_contact_afspraak_gaat,"[medicatie, contact, afspraak, gaat, graag, be...","[Beste [PERSOON], ik heb zojuist contact gehad..."
1,0,820,0_apotheek_recept_sturen_faxen,"[apotheek, recept, sturen, faxen, nieuw, ophal...",[Het recept kan naar Apotheek [LOCATIE] bij he...
2,1,688,1_huisarts_dokter_arts_medische,"[huisarts, dokter, arts, medische, contact, do...","[Of had de huisarts dit moeten doen?, Of moet ..."
3,2,599,2_humira_spuit_spuiten_zetten,"[humira, spuit, spuiten, zetten, injectie, ges...","[Dit alles had ik bij de humira niet., Ik heb ..."
4,3,597,3_afspraak_staan_plannen_afspraken,"[afspraak, staan, plannen, afspraken, verzette...",[Maandag heb ik weer een afspraak met [PERSOON...
...,...,...,...,...,...
250,249,26,249_snelle_reactie_respons_antwoord,"[snelle, reactie, respons, antwoord, bedankt, ...","[Bedankt voor de snelle reactie!, Bedankt voor..."
251,250,26,250_verhoging_gisteravond_middagavond_vroege,"[verhoging, gisteravond, middagavond, vroege, ...","[Heb er geen verhoging bij., Ik keijg er ook v..."
252,251,25,251_allergische_allergie_tabletje_geslikt,"[allergische, allergie, tabletje, geslikt, int...",[19 mei bij huisarts geweest en die dacht aan ...
253,252,25,252_inloggen_account_wachtwoord_inlog,"[inloggen, account, wachtwoord, inlog, invoere...","[Hallo, Dit is geen vraag over mijn klachten, ..."


In [8]:
topics = topic_model.topics_
new_topics = topic_model.reduce_outliers(sentences, topics)

In [9]:
topic_model.update_topics(sentences, topics=new_topics)

2025-06-01 15:52:40,108 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [21]:
topic_info = topic_model.get_topic_info()

In [22]:
topic_info = topic_info[1:]
topic_info.head()

,Topic,Count,Name,Representation,Representative_Docs
1,0,1044,0_recept_apotheek_locatie_sturen,"[recept, apotheek, locatie, sturen, nieuw, naa...",[Het recept kan naar Apotheek [LOCATIE] bij he...
2,1,827,1_huisarts_dokter_arts_geweest,"[huisarts, dokter, arts, geweest, bij, contact...","[Of had de huisarts dit moeten doen?, Of moet ..."
3,2,602,2_humira_spuit_spuiten_zetten,"[humira, spuit, spuiten, zetten, gebruik, inje...","[Dit alles had ik bij de humira niet., Ik heb ..."
4,3,926,3_afspraak_staan_gepland_afspraken,"[afspraak, staan, gepland, afspraken, verzette...",[Maandag heb ik weer een afspraak met [PERSOON...
5,4,538,4_ontlasting_dunne_dun_per,"[ontlasting, dunne, dun, per, dag, waterige, v...","[en had ik wel elke dag ontlasting., en heel v..."


In [23]:
len(topic_info)

254

In [25]:
topic_info_sorted = topic_info.sort_values('Count', ascending=False)

In [26]:
topic_info_sorted

,Topic,Count,Name,Representation,Representative_Docs
1,0,1044,0_recept_apotheek_locatie_sturen,"[recept, apotheek, locatie, sturen, nieuw, naa...",[Het recept kan naar Apotheek [LOCATIE] bij he...
4,3,926,3_afspraak_staan_gepland_afspraken,"[afspraak, staan, gepland, afspraken, verzette...",[Maandag heb ik weer een afspraak met [PERSOON...
2,1,827,1_huisarts_dokter_arts_geweest,"[huisarts, dokter, arts, geweest, bij, contact...","[Of had de huisarts dit moeten doen?, Of moet ..."
3,2,602,2_humira_spuit_spuiten_zetten,"[humira, spuit, spuiten, zetten, gebruik, inje...","[Dit alles had ik bij de humira niet., Ik heb ..."
15,14,566,14_medicatie_medicijnen_medicijn_stoppen,"[medicatie, medicijnen, medicijn, stoppen, geb...","[De medicatie is namelijk op., Mijn medicatie ..."
...,...,...,...,...,...
246,245,39,245_mtx_foliumzuur_bestellen_spuiten,"[mtx, foliumzuur, bestellen, spuiten, 20, will...","[Hallo, Ik zou graag mijn medicatie willen bes..."
243,242,36,242_advies_geruststelling_steun_herhalen,"[advies, geruststelling, steun, herhalen, hier...","[Bedankt voor het advies., Graag een advies va..."
252,251,34,251_allergische_allergie_tabletje_intoleranties,"[allergische, allergie, tabletje, intolerantie...",[19 mei bij huisarts geweest en die dacht aan ...
234,233,32,233_darmkanker_bevolkingsonderzoek_meegedaan_o...,"[darmkanker, bevolkingsonderzoek, meegedaan, o...","[Beste heer [PERSOON], Onlangs een brief ontva..."
